In [1]:
import pandas as pd
import os

DATA_WORK = r"C:\Users\33669\OneDrive\Документы\data_credit_scoring\tri_state_ai\data_work"

# Liste des fichiers HMDA à recharger proprement
years = range(2018, 2024)
files = [os.path.join(DATA_WORK, f"hmda_tristate_{y}.parquet") for y in years]

dfs = []
for f in files:
    print("📖 Lecture (SAFE) :", f)
    df = pd.read_parquet(f)

    # FORCER census_tract → STRING (intact)
    df["census_tract"] = df["census_tract"].astype(str).str.strip()

    # Retirer les ".0" introduits par PyArrow
    df["census_tract"] = df["census_tract"].str.replace(".0", "", regex=False)

    # zero-pad à 11 caractères si nécessaire
    df["census_tract"] = df["census_tract"].str.zfill(11)

    # Construire GEOID
    df["geoid_tract"] = df["census_tract"]

    # garder colonnes utiles pour le modèle
    keep = [
        "year", "state_code", "county_code", "census_tract", "geoid_tract",
        "loan_amount", "loan_purpose", "loan_type", "lien_status",
        "hoepa_status", "action_taken", "applicant_sex",
        "derived_ethnicity", "derived_race"
    ]

    df = df[keep]

    # variable cible
    df["approved"] = (df["action_taken"] == 1).astype(int)

    dfs.append(df)

# Fusionner toutes les années
hmda_model = pd.concat(dfs, ignore_index=True)

OUT = os.path.join(DATA_WORK, "hmda_tristate_model_2018_2023_FIXED.parquet")
hmda_model.to_parquet(OUT, index=False)

print("\n🎉 HMDA modèle corrigé créé !")
print("   →", OUT)
print("   Shape :", hmda_model.shape)



📖 Lecture (SAFE) : C:\Users\33669\OneDrive\Документы\data_credit_scoring\tri_state_ai\data_work\hmda_tristate_2018.parquet
📖 Lecture (SAFE) : C:\Users\33669\OneDrive\Документы\data_credit_scoring\tri_state_ai\data_work\hmda_tristate_2019.parquet
📖 Lecture (SAFE) : C:\Users\33669\OneDrive\Документы\data_credit_scoring\tri_state_ai\data_work\hmda_tristate_2020.parquet
📖 Lecture (SAFE) : C:\Users\33669\OneDrive\Документы\data_credit_scoring\tri_state_ai\data_work\hmda_tristate_2021.parquet
📖 Lecture (SAFE) : C:\Users\33669\OneDrive\Документы\data_credit_scoring\tri_state_ai\data_work\hmda_tristate_2022.parquet
📖 Lecture (SAFE) : C:\Users\33669\OneDrive\Документы\data_credit_scoring\tri_state_ai\data_work\hmda_tristate_2023.parquet

🎉 HMDA modèle corrigé créé !
   → C:\Users\33669\OneDrive\Документы\data_credit_scoring\tri_state_ai\data_work\hmda_tristate_model_2018_2023_FIXED.parquet
   Shape : (571820, 15)


In [2]:
import os
import pandas as pd

# ================================
# 0) PATHS
# ================================
BASE_DIR = r"C:\Users\33669\OneDrive\Документы\data_credit_scoring\tri_state_ai"
DATA_WORK = os.path.join(BASE_DIR, "data_work")
ACS_RAW   = os.path.join(BASE_DIR, "data_raw", "acs")

HMDA_FILE = os.path.join(DATA_WORK, "hmda_tristate_model_2018_2023_FIXED.parquet")
ACS_FILE  = os.path.join(ACS_RAW,  "acs5_tristate_2018_2023.parquet")

print("HMDA_MODEL_FILE :", HMDA_FILE)
print("ACS_MODEL_FILE  :", ACS_FILE)

# ================================
# 1) CHARGER LES DEUX BASES
# ================================
hmda = pd.read_parquet(HMDA_FILE)
acs  = pd.read_parquet(ACS_FILE)

print("\n📂 HMDA modèle 2018–2023 chargé :", hmda.shape)
print("Colonnes HMDA (aperçu) :", list(hmda.columns))

print("\n📂 ACS 2018–2023 chargé :", acs.shape)
print("Colonnes ACS (avant GEOID) :", list(acs.columns))

# ================================
# 2) CONSTRUIRE GEOID ACS (state+county+tract)
# ================================
# On s'assure que state / county / tract sont des strings bien padées
acs["state"]  = acs["state"].astype(str).str.zfill(2)
acs["county"] = acs["county"].astype(str).str.zfill(3)
acs["tract"]  = acs["tract"].astype(str).str.zfill(6)

acs["geoid_tract"] = acs["state"] + acs["county"] + acs["tract"]

# Sanity check : toutes les longueurs doivent faire 11
len_ok = acs["geoid_tract"].str.len().eq(11).mean() * 100
print(f"\n🔍 % GEOID ACS de longueur 11 : {len_ok:.4f} %")

print("\nExemples ACS après GEOID :")
print(acs[["year", "state", "county", "tract", "geoid_tract"]].head())

# ================================
# 3) RENAME VARIABLES ACS + CAST NUMÉRIQUE
# ================================
acs = acs.rename(columns={
    "B19013_001E": "acs_median_income",
    "B01003_001E": "acs_pop_total",
    "B02001_002E": "acs_white",
    "B02001_003E": "acs_black",
    "B02001_005E": "acs_asian",
    "B03003_003E": "acs_hispanic",
    "B23025_005E": "acs_unemployed",
    "B23025_003E": "acs_labor_force",
    "B17020_002E": "acs_poverty_num",
    "B17020_001E": "acs_poverty_den",
})

num_cols = [
    "acs_median_income", "acs_pop_total",
    "acs_white", "acs_black", "acs_asian", "acs_hispanic",
    "acs_unemployed", "acs_labor_force",
    "acs_poverty_num", "acs_poverty_den"
]

acs[num_cols] = acs[num_cols].apply(pd.to_numeric, errors="coerce")

# Ratios dérivés
acs["acs_poverty_rate"] = acs["acs_poverty_num"] / acs["acs_poverty_den"]
acs["acs_unemployment_rate"] = acs["acs_unemployed"] / acs["acs_labor_force"]
acs["acs_share_white"] = acs["acs_white"]       / acs["acs_pop_total"]
acs["acs_share_black"] = acs["acs_black"]       / acs["acs_pop_total"]
acs["acs_share_asian"] = acs["acs_asian"]       / acs["acs_pop_total"]
acs["acs_share_hispanic"] = acs["acs_hispanic"] / acs["acs_pop_total"]

print("\n✅ ACS model – types numériques + ratios créés.")
print(acs.head()[[
    "NAME", "acs_median_income", "acs_pop_total",
    "acs_poverty_rate", "acs_unemployment_rate",
    "acs_share_white", "acs_share_black", "acs_share_asian", "acs_share_hispanic",
    "year", "geoid_tract"
]])

print("\n📦 ACS_MODEL prêt pour merge :", acs.shape)

# ================================
# 4) SANITY CHECK CÔTÉ HMDA
# ================================
print("\nExemples HMDA geoid_tract :")
print(hmda[["year", "state_code", "county_code", "census_tract", "geoid_tract"]].head())

len_ok_hmda = hmda["geoid_tract"].astype(str).str.len().eq(11).mean() * 100
print(f"\n🔍 % GEOID HMDA de longueur 11 : {len_ok_hmda:.4f} %")

# ================================
# 5) MERGE FINAL HMDA + ACS
# ================================
hmda_acs = hmda.merge(
    acs,
    on=["year", "geoid_tract"],
    how="left"
)

OUT = os.path.join(DATA_WORK, "hmda_acs_tristate_2018_2023_FINAL.parquet")
hmda_acs.to_parquet(OUT, index=False)

print("\n🎉 MERGE HMDA + ACS TERMINÉ !")
print("   →", OUT)
print("   → Shape :", hmda_acs.shape)

# ================================
# 6) COUVERTURE ACS PAR ANNÉE
# ================================
coverage = (
    hmda_acs
    .assign(hasACS=~hmda_acs["acs_median_income"].isna())
    .groupby("year")
    .agg(
        coverage_pct=("hasACS", lambda s: s.mean() * 100),
        n_loans=("hasACS", "size")
    )
)

print("\n📊 Couverture ACS par année (via acs_median_income) :")
print(coverage)

# Petit résumé global
n_tract_hmda = hmda["geoid_tract"].nunique()
n_tract_acs  = acs["geoid_tract"].nunique()
n_tract_match = hmda_acs.loc[~hmda_acs["acs_median_income"].isna(), "geoid_tract"].nunique()

print(f"\n🏘️ Nombre de tracts distincts HMDA 2018–2023 : {n_tract_hmda}")
print(f"🏘️ Nombre de tracts distincts ACS  2018–2023 : {n_tract_acs}")
print(f"🏘️ Nombre de tracts matchés HMDA+ACS        : {n_tract_match}")
print(f"➡️ Couverture en tracts (matchés / HMDA) : {n_tract_match / n_tract_hmda * 100:.2f} %")

HMDA_MODEL_FILE : C:\Users\33669\OneDrive\Документы\data_credit_scoring\tri_state_ai\data_work\hmda_tristate_model_2018_2023_FIXED.parquet
ACS_MODEL_FILE  : C:\Users\33669\OneDrive\Документы\data_credit_scoring\tri_state_ai\data_raw\acs\acs5_tristate_2018_2023.parquet

📂 HMDA modèle 2018–2023 chargé : (571820, 15)
Colonnes HMDA (aperçu) : ['year', 'state_code', 'county_code', 'census_tract', 'geoid_tract', 'loan_amount', 'loan_purpose', 'loan_type', 'lien_status', 'hoepa_status', 'action_taken', 'applicant_sex', 'derived_ethnicity', 'derived_race', 'approved']

📂 ACS 2018–2023 chargé : (49409, 24)
Colonnes ACS (avant GEOID) : ['NAME', 'B19013_001E', 'B01003_001E', 'B02001_002E', 'B02001_003E', 'B02001_005E', 'B03003_003E', 'B23025_005E', 'B23025_003E', 'B17020_002E', 'B17020_001E', 'B15003_017E', 'B15003_018E', 'B15003_019E', 'B15003_020E', 'B15003_021E', 'B15003_022E', 'B15003_023E', 'B15003_024E', 'B15003_025E', 'state', 'county', 'tract', 'year']

🔍 % GEOID ACS de longueur 11 : 100.

In [3]:
FINAL_COLS = [
    'year', 'state_code', 'county_code', 'census_tract', 'geoid_tract',
    'loan_amount', 'loan_purpose', 'loan_type', 'lien_status',
    'hoepa_status', 'action_taken', 'applicant_sex',
    'derived_ethnicity', 'derived_race', 'approved',
    'NAME', 'acs_median_income', 'acs_pop_total',
    'acs_white', 'acs_black', 'acs_asian', 'acs_hispanic',
    'acs_unemployed', 'acs_labor_force',
    'acs_poverty_num', 'acs_poverty_den',
    'state', 'county', 'tract',
    'acs_poverty_rate', 'acs_unemployment_rate',
    'acs_share_white', 'acs_share_black',
    'acs_share_asian', 'acs_share_hispanic'
]

hmda_acs = hmda_acs[FINAL_COLS].copy()

OUT = os.path.join(DATA_WORK, "hmda_acs_tristate_2018_2023_FINAL.parquet")
hmda_acs.to_parquet(OUT, index=False)

print("✅ Fichier FINAL corrigé sauvegardé :", OUT)
print("Shape :", hmda_acs.shape)
print("Colonnes :", hmda_acs.columns.tolist())




✅ Fichier FINAL corrigé sauvegardé : C:\Users\33669\OneDrive\Документы\data_credit_scoring\tri_state_ai\data_work\hmda_acs_tristate_2018_2023_FINAL.parquet
Shape : (571820, 35)
Colonnes : ['year', 'state_code', 'county_code', 'census_tract', 'geoid_tract', 'loan_amount', 'loan_purpose', 'loan_type', 'lien_status', 'hoepa_status', 'action_taken', 'applicant_sex', 'derived_ethnicity', 'derived_race', 'approved', 'NAME', 'acs_median_income', 'acs_pop_total', 'acs_white', 'acs_black', 'acs_asian', 'acs_hispanic', 'acs_unemployed', 'acs_labor_force', 'acs_poverty_num', 'acs_poverty_den', 'state', 'county', 'tract', 'acs_poverty_rate', 'acs_unemployment_rate', 'acs_share_white', 'acs_share_black', 'acs_share_asian', 'acs_share_hispanic']


In [4]:
import os
import pandas as pd

BASE_DIR = r"C:\Users\33669\OneDrive\Документы\data_credit_scoring\tri_state_ai\data_work"

CORE_PATH  = os.path.join(BASE_DIR, "hmda_tristate_core_2007_2024_v2.parquet")
MODEL_PATH = os.path.join(BASE_DIR, "hmda_tristate_model_2018_2023_FIXED.parquet")
ACS_PATH   = os.path.join(BASE_DIR, "hmda_acs_tristate_2018_2023_FINAL.parquet")

core = pd.read_parquet(CORE_PATH)
model = pd.read_parquet(MODEL_PATH)
hmda_acs = pd.read_parquet(ACS_PATH)

print("core shape :", core.shape)
print("model shape:", model.shape)
print("acs shape  :", hmda_acs.shape)

core shape : (14163286, 12)
model shape: (571820, 15)
acs shape  : (571820, 35)


In [5]:
def quick_cols(df, name, n=30):
    print(f"\n=== {name} ===")
    print(list(df.columns[:n]))

quick_cols(core, "CORE")
quick_cols(model, "MODEL")
quick_cols(hmda_acs, "HMDA+ACS")


=== CORE ===
['action_taken', 'applicant_sex', 'county_code', 'hoepa_status', 'lien_status', 'loan_purpose', 'loan_type', 'preapproval', 'purchaser_type', 'rate_spread', 'state_code', 'year']

=== MODEL ===
['year', 'state_code', 'county_code', 'census_tract', 'geoid_tract', 'loan_amount', 'loan_purpose', 'loan_type', 'lien_status', 'hoepa_status', 'action_taken', 'applicant_sex', 'derived_ethnicity', 'derived_race', 'approved']

=== HMDA+ACS ===
['year', 'state_code', 'county_code', 'census_tract', 'geoid_tract', 'loan_amount', 'loan_purpose', 'loan_type', 'lien_status', 'hoepa_status', 'action_taken', 'applicant_sex', 'derived_ethnicity', 'derived_race', 'approved', 'NAME', 'acs_median_income', 'acs_pop_total', 'acs_white', 'acs_black', 'acs_asian', 'acs_hispanic', 'acs_unemployed', 'acs_labor_force', 'acs_poverty_num', 'acs_poverty_den', 'state', 'county', 'tract', 'acs_poverty_rate']
